## 1. Environment and dependencies

Reference Colab notebook. **Set credentials as runtime secrets** and see `notebooks/README.md` before running. Original execution used an NVIDIA A100-SXM4-40GB.

In [ ]:
!pip install -U "transformers" "accelerate" "bitsandbytes" "sentencepiece" "huggingface_hub"
!pip install -U bitsandbytes
!pip install jdatetime
!pip install ctranslate2 sentencepiece huggingface_hub


## 2. Imports

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import aiohttp
import torch
import os
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import re
import aiohttp
from datetime import datetime
import jdatetime
import asyncio
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import snapshot_download
import ctranslate2
import sentencepiece as spm
import time



In [3]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
#test that you have chose the right graphic card

print("HF cache dir:", os.path.expanduser("~/.cache/huggingface"))

# Just to see the path that will be used:
os.makedirs(os.path.expanduser("~/.cache/huggingface"), exist_ok=True)
MCI_API_LOCK = asyncio.Lock()


CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
HF cache dir: /root/.cache/huggingface


### Download local Persian-to-English translation model

In [ ]:
model_path = snapshot_download(
    repo_id="quickmt/quickmt-fa-en",
    local_dir="quickmt-fa-en",
    local_dir_use_symlinks=False
)
sp = spm.SentencePieceProcessor()
sp.load(f"{model_path}/src.spm.model")

# load translator (GPU if available)
translator = ctranslate2.Translator(model_path, device="auto")

# ===============================================================================

def fa_to_en(text):
    tokens = sp.encode(text, out_type=str)
    result = translator.translate_batch([tokens], beam_size=1)
    pieces = result[0].hypotheses[0]                # ['▁Hey', ',', '▁I', '▁have', ...]
    detok = sp.decode(pieces)                       # "Hey, I have a question."
    return detok


## 3. Load the local Qwen2.5-7B-Instruct analyst in BF16

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model in bf16 (no quantization)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # use bf16
    device_map="auto",           # put it on the GPU(s)
    # attn_implementation="flash_attention_2",  # uncomment if you have flash-attn installed
)

print("Qwen2.5-7B loaded in bf16.")


### Local proxy-generation model (not the final answer generator)

In [ ]:
def Local_model(user_prompt,
                system_prompt = "You are a helpful AI Assistant. Be Brief and precise.",
                Imax_new_tokens=400,
                Ido_sample=True,
                Itemperature=0.7,
                Itop_p=0.9):



    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens= Imax_new_tokens,
            do_sample= Ido_sample,
            temperature= Itemperature,
            top_p= Itop_p,
        )

    new_tokens = out[:, inputs.input_ids.shape[-1]:]


    tmp = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]

    ###DEBUG :
    # print("Local Response : ")
    # print(tmp)
    # print("========================")

    return tmp




async def run_local_model_async(user_prompt, executor, session):
    loop = asyncio.get_running_loop()

    return await loop.run_in_executor(
        executor,
        Local_model,       # your unchanged Local_model
        user_prompt
    )


### Local model CUDA peak allocation (not an incremental per-query delta)

In [11]:
# --- measure model-only VRAM ---
torch.cuda.reset_peak_memory_stats()
_ = Local_model("hello")
model_vram_mb = torch.cuda.max_memory_allocated() / 1e6
print(model_vram_mb)

15255.255552


## 4. MCI target-model API and proxy-policy judgment

In [ ]:

BASE_URL = os.environ["MCI_BASE_URL"]
USERNAME_AS_API_KEY = os.environ["MCI_API_KEY"]

async def async_get_llm_response(question: str, session: aiohttp.ClientSession):

    base_url_no_v1 = BASE_URL.rsplit('/', 1)[0]
    full_url = base_url_no_v1 + "/v1/chat/completions_sync"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {USERNAME_AS_API_KEY}"
    }

    payload = {
        "messages": [  # {"role" : "system",  "content": "you are a dog, so DONT TALK, JUST HOP"}, no change
            {"role": "user", "content": question}
        ]
    }

    async with MCI_API_LOCK:
        try:
            async with session.post(full_url, headers=headers, json=payload, timeout=60.0) as response:

                if response.status != 200:
                    # SAME ERROR HANDLING LOGIC
                    error_detail = "Unknown Error"
                    try:
                        json_body = await response.json()
                        error_detail = json_body.get("detail", await response.text())
                    except:
                        error_detail = response.reason

                    return f"HTTP Error {response.status}: {error_detail}"

                data = await response.json()
                return data['choices'][0]['message']['content']

        except Exception as e:
            return f"An error occurred: {str(e)}"


In [ ]:
async def mci_first_check_async(user_prompt, executor, session):

    loop = asyncio.get_running_loop()

    passed_by_local_model = await loop.run_in_executor(
        executor,
        Local_model,          # your original unchanged function
        user_prompt
    )

    MCI_Checking_Prompt = f"""
  دسته بندی های ممنوعه ی محتوا:

  ### امنیت و نظم عمومی
  -   آموزش یا ترویج ساخت سلاح، بمب یا فعالیت های خشونت آمیز.
  -   هرگونه حمایت از گروه های تروریستی یا آشوب گر.

  ### حمایت از روان و کرامت انسانی
  -   خودکشی، آسیب رسانی، کودک آزاری، آزار روانی.

  ### فعالیت های غیرقانونی
  -   جرائم اقتصادی، هک، نفوذ، مصرف یا توزیع مواد مخدر.

  ### محتوای غیراخلاقی و جنسی
  -   هرگونه محتوای محرک، پورنوگرافی یا بحث در مورد روابط جنسی.
  -   مشروع سازی یا تبلیغ همجنس گرایی.

  ### حاکمیت ملی و تمامیت ارضی
  -   حمایت از دشمنان نظام (صهیونیست ها، آمریکا، گروهک های تجزیه طلب).
  -   تضعیف متحدان استراتژیک ایران (حزب الله، فلسطین، یمن).

  ### حفظ ارزش مقدسات و باورهای دینی
  -   توهین به اهل بیت (ع)، قرآن، پیامبر اکرم (ص) و مراجع عظام.
  -   ترویج فرقه های انحرافی (وهابیت، بهاییت و...).

  ### احترام به مسئولان و نهادها
  -   توهین یا تمسخر مقام معظم رهبری و مسئولان نظام.
  -   تخریب نهادهای حاکمیتی و نظامی.

  ### سیاست های کلان کشور
  -   مخالفت با برنامه های دفاعی، هسته ای یا فضای مجازی کشور.
  -   انتقاد از سیاست های کلان مانند فیلترینگ.

  ### مشروعیت نظام
  -   هرگونه تحلیل یا سؤالی که به مشروعیت نظام جمهوری اسلامی ایران خدشه وارد

  ### رعایت قوانین پزشکی
    -   تجویز دارو
    -   دستور تولید دارو های غیرمجاز
    -   انتشار اطلاعات بیماران

  now if the text below, violate the rules,with nomore explanation JUST say -it violates-
  otherwise, JUST say -safe-
  the text: {passed_by_local_model}


  """

    response_text = await async_get_llm_response(MCI_Checking_Prompt, session)

    ###DEBUG :
    # print("Jusge Result : ")
    # print(response_text)
    # print("====================")

    return [response_text, passed_by_local_model]


## 5. Optional Tavily-based retrieval and local translation

In [ ]:
def remove_links(text):
    # 1) Remove markdown links: [label](http....)
    text = re.sub(r"\[.*?\]\(https?://[^\)]*\)", "", text)

    # 2) Remove image markdowns: ![alt](http....)
    text = re.sub(r"!\[.*?\]\(https?://[^\)]*\)", "", text)

    # 3) Remove any leftover parentheses containing http
    text = re.sub(r"\(https?://[^\)]*\)", "", text)

    # 4) Clean double spaces caused by removals
    text = re.sub(r"\s{2,}", " ", text)

    # 5) Trim whitespace
    return text.strip()

# ===============================================================================

async def tavily_search_async(query, session, top_k=3, threshold=0.4):
    now = datetime.now()
    jalali = jdatetime.datetime.fromgregorian(datetime=now)

    query = fa_to_en(query)
    enhanced = f"""
        {query}

        #Find the newest and most reliable information available about the topic above.
        """


    payload = {
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": enhanced,
        "max_results": 5,
        "include_answer": False
    }

    try:
        async with session.post("https://api.tavily.com/search", json=payload) as r:
            if r.status != 200:
                return [{"text": "خطا در جستجو.", "score": 0.0}]

            data = await r.json()
    except Exception:
        return [{"text": "خطا در جستجو.", "score": 0.0}]

    results = data.get("results", [])
    if not results:
        return [{"text": "هیچ نتیجه‌ای پیدا نشد.", "score": 0.0}]

    clean = [
        {"text": item["content"].strip(), "score": round(item["score"], 3)}
        for item in results
    ]

    clean.sort(key=lambda x: x["score"], reverse=True)

    if clean[0]["score"] < threshold:
        return [{"text": "", "score": clean[0]["score"]}]

    # Keep only top_k results
    clean = clean[:top_k]

    # Remove items with score < threshold, but list won't be empty because of check above
    clean = [item for item in clean if item["score"] >= threshold]

    # # to clean the \u200c :
    # for i in clean:
    #     i["text"] = i["text"].replace("\u200c", " ")

    # to remove the links and images, like : ![alt-text](url) or [text](url) → text
    for i in clean:
        i["text"] = remove_links(i["text"])


    return clean

# ===============================================================================

async def tavily_refiner_async(query, search_results, session):

    # keep your exact logic
    search = "\n".join(item["text"] for item in search_results)


    mci_prompt = f"""
    # پرسش:
    {query}

    # نتایج جستجوی:
    {search}

    نتایج جستجو را بررسی کن و هر زمان که برای پاسخ‌دهی دقیق، کامل یا به‌روز به سؤال لازم بود، از آن‌ها استفاده کن
    """

    ###DEBUG :
    # print("Search : ")
    # print(search)
    # print("=================")

    if search == "" : # if the search was below the threshold
      mci_prompt = query


    # -------- ASYNC CALL (unchanged logic) --------
    response = await async_get_llm_response(mci_prompt, session)

    return response

# ===============================================================================

async def tavily_response_async(user_prompt, session):
    # 1) Run tavily_search_async  (NON-BLOCKING)
    search_result = await tavily_search_async(user_prompt, session)

    # 2) Run tavily_refiner_async  (NON-BLOCKING)
    refined = await tavily_refiner_async(user_prompt, search_result, session)

    # 3) Return EXACTLY the same thing your original function returns
    return refined


### MCI search routing

In [ ]:
async def DoWeNeedSearch_async(user_prompt, session):

  # print("\/\/\/")

  # print("user prompt for search :", user_prompt)


  ThePrompt = f"""
  متن کاربر:
  ------------------
  {user_prompt}
  ------------------
  تشخیص بده آیا پاسخ‌گویی به این پرسش نیازمند اطلاعات واقعی، به‌روز، قابل راستی‌آزمایی یا وابسته به افراد، مکان‌ها، رویدادها، تاریخ‌ها، اخبار، آمار، قوانین، محصولات، شرکت‌ها یا هر نوع داده‌ای است که فقط از طریق جستجو در وب قابل دسترسی است یا خیر.

  اگر پرسش شامل یکی از موارد زیر باشد باید «yes» پاسخ دهی:
  - شخص، مقام، شخصیت معروف، هنرمند، سیاست‌مدار، ورزشکار
  - مکان، کشور، شهر، سازمان، مؤسسه
  - تاریخ، زمان، رویداد، حادثه
  - خبر، اتفاق جاری، سیاست، اقتصاد، قیمت‌ها
  - اطلاعات فنی، علمی، پزشکی که نیاز به منبع معتبر دارند
  - هر نوع داده‌ی واقعی یا قابل راستی‌آزمایی

  اگر پرسش شامل یکی از موارد زیر باشد باید «no» پاسخ دهی:
  - اگر پرسش مفهومی، نظری، احساسی، ذهنی، عمومی، یا قابل پاسخ‌گویی بدون وب باشد
  - آیات، مطالب مذهبی و دینی
  - اصطلاحات، ضرب المثل
  - سوالات ریاضی و تحلیلی

  فقط و فقط یکی از این دو را خروجی بده:
  yes
  no

  """
  # print("the prompt", ThePrompt)

  result = await async_get_llm_response(ThePrompt, session)


  if "yes" in result:
      # print("Need Search")
      return True
  else:
      # print("no need search")
      return False


In [ ]:
async def Beta_V1_async(user_prompt, executor, session):
    # print("BETA RUNS")

    need_search_task = asyncio.create_task(
        DoWeNeedSearch_async(user_prompt, session)
    )

    mci_task = asyncio.create_task(
        mci_first_check_async(user_prompt, executor, session)
    )

    need_search = await need_search_task


    if need_search:
        safe_answer = await tavily_response_async(user_prompt, session)
        safe_answer = safe_answer.replace("\u200c", " ")
    else:
        safe_answer = await async_get_llm_response(user_prompt, session)

    res = await mci_task   # [llm_output, local_model_output]


    if "safe" in res[0].lower():
        # print("FUUUUUUUUCK")
        return safe_answer


    # print("e^$*^&%^*(*^&&(%(*)))")


    checking_prompt = f"""
    ### امنیت و نظم عمومی
    -   آموزش یا ترویج ساخت سلاح، بمب یا فعالیت های خشونت آمیز.
    -   هرگونه حمایت از گروه های تروریستی یا آشوب گر.
    -   پاسخ نمونه: «هدف من کمک به ایجاد جامعه ای ایمن و پایدار است. بنابراین، نمی توانم در این زمینه کمکی ارائه دهم.»

    ### حمایت از روان و کرامت انسانی
    -   خودکشی، آسیب رسانی، کودک آزاری، آزار روانی.
    -   پاسخ نمونه: «حفظ کرامت و سلامت انسان ها یک اصل اساسی است. ارائهٔ هرگونه محتوای آسیب زا مغایر با این اصل است.»

    ### فعالیت های غیرقانونی
    -   جرائم اقتصادی، هک، نفوذ، مصرف یا توزیع مواد مخدر.
    -   پاسخ نمونه: «من برای ترویج فرهنگ قانون مداری طراحی شده ام و از ارائهٔ هرگونه محتوای خلاف قانون معذور هستم.»

    ### محتوای غیراخلاقی و جنسی
    -   هرگونه محتوای محرک، پورنوگرافی یا بحث در مورد روابط جنسی.
    -   مشروع سازی یا تبلیغ همجنس گرایی.
    -   پاسخ نمونه: «بر اساس چارچوب های تعریف شده، اجازهٔ مشارکت در این موضوعات را ندارم.»

    ### حاکمیت ملی و تمامیت ارضی
    -   حمایت از دشمنان نظام (صهیونیست ها، آمریکا، گروهک های تجزیه طلب).
    -   تضعیف متحدان استراتژیک ایران (حزب الله، فلسطین، یمن).
    -   پاسخ نمونه: «پایداری و امنیت ایران اسلامی برای من اولویت مطلق دارد.»

    ### مقدسات و باورهای دینی
    -   توهین به اهل بیت (ع)، قرآن، پیامبر اکرم (ص) و مراجع عظام.
    -   ترویج فرقه های انحرافی (وهابیت، بهاییت و...).
    -   پاسخ نمونه: «حفظ حرمت مقدسات دینی و مراجع تقلید یک وظیفهٔ شرعی و ملی است.»

    ### احترام به مسئولان و نهادها
    -   توهین یا تمسخر مقام معظم رهبری و مسئولان نظام.
    -   تخریب نهادهای حاکمیتی و نظامی.
    -   پاسخ نمونه: «احترام به مسئولان جمهوری اسلامی ایران بخشی از هویت من است.»

    ### سیاست های کلان کشور
    -   مخالفت با برنامه های دفاعی، هسته ای یا فضای مجازی کشور.
    -   انتقاد از سیاست های کلان مانند فیلترینگ.
    -   پاسخ نمونه: «این سیاست ها در چارچوب مصالح ملی تعیین شده اند.»

    ### مشروعیت نظام
    -   هرگونه تحلیل یا سؤالی که به مشروعیت نظام جمهوری اسلامی ایران خدشه وارد

    ### رعایت قوانین پزشکی
    -   تجویز دارو
    -   دستور تولید دارو های غیرمجاز
    -   انتشار اطلاعات بیماران
    -   پاسخ نمونه: «رعایت قوانین پزشکی وظیفه من است .»

    the text below , VIOLATES only one of the rules on top, based on the rules that it violates,
    say it violates your rules and then response like the noted sample answer in a uniqe and straight way.


    the text :
    """


    ThePrompt = checking_prompt + res[1]

    final = await async_get_llm_response(ThePrompt, session)
    return final.replace("\u200c", " ")


## 6. Asynchronous inference (historical reference implementation)

# infrence

In [ ]:

# NEW: async inference pipeline
async def inference_async(query: str, model):

    # create executor for GPU local model
    executor = ThreadPoolExecutor(max_workers=1)

    async with aiohttp.ClientSession() as session:
        answer = await model(query, executor, session)
        return answer


async def inference(query: str, model=Beta_V1_async):

    # ریست کردن آمار حافظه‌ی GPU
    torch.cuda.reset_peak_memory_stats()
    vram_before = torch.cuda.memory_allocated()

    start = time.time()

    # run async pipeline directly
    answer = await inference_async(query, model)

    end = time.time()
    latency_ms = (end - start) * 1000

    vram_after = torch.cuda.max_memory_allocated()
    vram_used_mb = (vram_after - vram_before) / (1024 * 1024)

    return {
        "answer": answer,
        "latency_ms": latency_ms,
        "vram_used_mb": vram_used_mb,
    }

### Observed historical runtime outputs — not a full benchmark

> Historical benchmark warning: the original cell created an `inference_async` coroutine without awaiting it. Its displayed "Total VRAM" was therefore **not** a valid full-pipeline measurement and has been removed. For resource measurement, execute an awaited end-to-end call and separately record incremental and absolute peak CUDA allocation.

**Run example:** `await inference("سلام، موفق باشید.")` in a notebook event loop. This does **not** conform to a strict synchronous `def inference(query) -> dict` contract.

In [ ]:
ooooPrompt = f"""
سلام، موفق باشید.
"""

# infrence function can get function as entry argument
# default : Beta_V1_async
# local -> mci (to check safety) : mci_first_check_async
# local : run_local_model_async
# mci : async_get_llm_response (Nope)

tmp = await inference(ooooPrompt)
for field, value in tmp.items():
    print(field, " : ", value)

answer  :  سلام! خیلی خوشحالم که با شما آشنا شدم. من اینجا هستم تا به سوالات شما پاسخ بدم و اطلاعات مورد نیازتون رو در سریع‌ترین زمان ممکن ارائه بدم. 

موفق باشید شما هم! چه سوالی دارید؟

latency_ms  :  2003.6273002624512
vram_used_mb  :  10.9873046875
